# NB1 — Preprocessing

v7 pipeline. Runs on Kaggle CPU.

Produces, per dataset, four disjoint splits:

- `{ds}_train.parquet`        — used for model fitting
- `{ds}_val.parquet`          — used for early stopping (NB2/NB3)
- `{ds}_test_balanced.parquet` — balanced XAI analysis set (1000 default / 1000 non-default), used for the main
  minority/majority analysis (RQ2) and for all XAI/faithfulness computations
- `{ds}_test_natural.parquet` — held-out set that preserves the dataset's natural class prevalence,
  used only for calibration metrics (ECE/NLL/Brier) that are not distorted by test-set rebalancing
  (see audit note C9: calibration computed on a 50:50 test set is not interpretable against a model
  trained on the natural, imbalanced prior).

All four splits are index-disjoint (checked with assertions). Imputation (median) and scaling
(standardization) are fit on `train` only and applied to the other three splits, to avoid leakage.

Upload the output directory as a Kaggle Dataset named `xai-credit-preprocessed`; it is the input
for NB2, NB3, NB4, NB5.


## 1. Installs & imports

In [ ]:
!pip install ucimlrepo scikit-learn pandas numpy pyarrow fastparquet xlrd kaggle -q

import os
import random
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import urllib.request
import zipfile

print("Imports completed.")


## 2. Configuration & utilities

In [ ]:
class Config:
    # --- Paths ---
    OUTPUT_DIR = "/kaggle/working"

    # --- Settings ---
    SEED = 42
    TEST_SIZE = 2000           # balanced XAI analysis set: 1000 positive + 1000 negative
    # v7: was 500. NB8's validation puts the minimum detectable USFG effect at 0.182 SD for
    # n=2500, and MDE scales as 1/sqrt(n) -- at n=500 that is ~0.41 SD, larger than nearly
    # every effect this study measures, which would make almost every negative result
    # uninterpretable. 2000 brings the MDE back to ~0.20 SD. Feasible on all three datasets:
    # the balanced split needs 1000 positives and GMSC has ~10k, Taiwan ~6.6k, HC ~24k.
    NATURAL_TEST_SIZE = 2000   # held-out set preserving natural class prevalence
    VAL_FRACTION = 0.15        # fraction of train_full used for validation


def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


seed_everything(Config.SEED)


## 3. Split functions

Two holdouts are carved out of the full dataset before any train/val split:

1. A **balanced** test set (`create_stratified_test`), unchanged from the original design — needed
   for the minority/majority analysis, since a natural-prevalence test set would contain too few
   default cases on datasets like Home Credit (~8% default rate) to support a reliable per-class
   Spearman correlation.
2. A **natural-prevalence** holdout (`create_natural_holdout`), carved out of what remains after
   step 1, sampled to match the dataset's true class ratio. Used only for calibration reporting.

Both holdouts are removed before the train/val split, so neither can leak into model fitting.


In [ ]:
def create_stratified_test(y, n_test=2000, seed=42):
    """Balanced test set: exactly n_test // 2 positive + n_test // 2 negative."""
    idx_pos = np.where(y == 1)[0]
    idx_neg = np.where(y == 0)[0]

    assert len(idx_pos) >= n_test // 2, "Not enough positive samples for balanced test"
    assert len(idx_neg) >= n_test // 2, "Not enough negative samples for balanced test"

    rng = np.random.RandomState(seed)
    test_pos = rng.choice(idx_pos, size=n_test // 2, replace=False)
    test_neg = rng.choice(idx_neg, size=n_test // 2, replace=False)

    test_idx = np.concatenate([test_pos, test_neg])
    remaining_idx = np.setdiff1d(np.arange(len(y)), test_idx)

    rng.shuffle(test_idx)
    rng.shuffle(remaining_idx)
    return remaining_idx, test_idx


def create_natural_holdout(y, remaining_idx, n_natural=2000, seed=42):
    """
    Held-out set drawn from remaining_idx that preserves the dataset's natural class rate.
    Sampled without replacement, stratified to the empirical prevalence of `remaining_idx`
    so the holdout's prevalence matches the population prevalence rather than being a coin flip.
    """
    rng = np.random.RandomState(seed + 1)
    rem_y = y[remaining_idx]
    rate = rem_y.mean()

    idx_pos = remaining_idx[rem_y == 1]
    idx_neg = remaining_idx[rem_y == 0]

    n_pos = int(round(n_natural * rate))
    n_neg = n_natural - n_pos
    n_pos = min(n_pos, len(idx_pos))
    n_neg = min(n_neg, len(idx_neg))

    nat_pos = rng.choice(idx_pos, size=n_pos, replace=False)
    nat_neg = rng.choice(idx_neg, size=n_neg, replace=False)
    natural_idx = np.concatenate([nat_pos, nat_neg])
    rng.shuffle(natural_idx)

    train_full_idx = np.setdiff1d(remaining_idx, natural_idx)
    return train_full_idx, natural_idx


## 4. Main processing function

In [ ]:
def process_and_save(df, target_col, dataset_name):
    print(f"\n{'=' * 60}")
    print(f"Processing {dataset_name}")
    print(f"{'=' * 60}")

    df = df.dropna(subset=[target_col])
    y = df[target_col].values
    X_df = df.drop(columns=[target_col])

    numeric_cols = X_df.select_dtypes(include=[np.number]).columns
    X_numeric = X_df[numeric_cols]

    print(f"Raw shape: {df.shape}. Numeric features: {X_numeric.shape[1]}")
    print(f"Overall default rate: {y.mean():.4f}")

    # --- 1. balanced test ---
    remaining_idx, test_idx = create_stratified_test(y, n_test=Config.TEST_SIZE, seed=Config.SEED)

    # --- 2. natural-prevalence holdout, carved out of what remains ---
    train_full_idx, natural_idx = create_natural_holdout(
        y, remaining_idx, n_natural=Config.NATURAL_TEST_SIZE, seed=Config.SEED
    )

    # --- 3. train / val split of what remains ---
    y_train_full = y[train_full_idx]
    train_idx, val_idx = train_test_split(
        train_full_idx, test_size=Config.VAL_FRACTION, stratify=y_train_full,
        random_state=Config.SEED,
    )

    # --- leakage assertions: all four splits pairwise disjoint ---
    splits = {"train": train_idx, "val": val_idx, "test_balanced": test_idx,
              "test_natural": natural_idx}
    names = list(splits.keys())
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            overlap = len(set(splits[a]).intersection(set(splits[b])))
            assert overlap == 0, f"Leakage: {a} and {b} overlap by {overlap} indices!"
    print("Leakage assertions passed: train / val / test_balanced / test_natural are "
          "pairwise disjoint.")

    # --- slice raw data ---
    X_train_raw = X_numeric.iloc[train_idx]
    X_val_raw = X_numeric.iloc[val_idx]
    X_test_raw = X_numeric.iloc[test_idx]
    X_natural_raw = X_numeric.iloc[natural_idx]

    y_train, y_val = y[train_idx], y[val_idx]
    y_test, y_natural = y[test_idx], y[natural_idx]

    # --- impute (median) + scale, fit on train only ---
    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train_raw)
    X_val_imp = imputer.transform(X_val_raw)
    X_test_imp = imputer.transform(X_test_raw)
    X_natural_imp = imputer.transform(X_natural_raw)

    # --- clip extreme outliers, fit on train only (v7 audit: some raw columns carry
    # data-entry errors or sentinel values -- e.g. GMSC's RevolvingUtilizationOfUnsecured-
    # Lines has entries in the tens of thousands where the field is defined on [0, ~2], and
    # Home Credit's FLAG_MOBIL is a near-constant flag -- that standardize to |z| > 100-500.
    # Percentile clipping bounds their influence without discarding the columns.) ---
    clip_lo = np.percentile(X_train_imp, 0.1, axis=0)
    clip_hi = np.percentile(X_train_imp, 99.9, axis=0)
    X_train_imp = np.clip(X_train_imp, clip_lo, clip_hi)
    X_val_imp = np.clip(X_val_imp, clip_lo, clip_hi)
    X_test_imp = np.clip(X_test_imp, clip_lo, clip_hi)
    X_natural_imp = np.clip(X_natural_imp, clip_lo, clip_hi)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_val_scaled = scaler.transform(X_val_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    X_natural_scaled = scaler.transform(X_natural_imp)

    def to_df(X_scaled, y_arr):
        d = pd.DataFrame(X_scaled, columns=numeric_cols)
        d["TARGET"] = y_arr
        return d

    df_train = to_df(X_train_scaled, y_train)
    df_val = to_df(X_val_scaled, y_val)
    df_test = to_df(X_test_scaled, y_test)
    df_natural = to_df(X_natural_scaled, y_natural)

    paths = {
        "train": f"{Config.OUTPUT_DIR}/{dataset_name}_train.parquet",
        "val": f"{Config.OUTPUT_DIR}/{dataset_name}_val.parquet",
        "test_balanced": f"{Config.OUTPUT_DIR}/{dataset_name}_test_balanced.parquet",
        "test_natural": f"{Config.OUTPUT_DIR}/{dataset_name}_test_natural.parquet",
    }
    df_train.to_parquet(paths["train"], index=False)
    df_val.to_parquet(paths["val"], index=False)
    df_test.to_parquet(paths["test_balanced"], index=False)
    df_natural.to_parquet(paths["test_natural"], index=False)

    print(f"train        : {df_train.shape[0]:>7} rows  -> {paths['train']}")
    print(f"val          : {df_val.shape[0]:>7} rows  -> {paths['val']}")
    print(f"test_balanced: {df_test.shape[0]:>7} rows  -> {paths['test_balanced']}  "
          f"(class balance: {df_test['TARGET'].mean():.3f})")
    print(f"test_natural : {df_natural.shape[0]:>7} rows  -> {paths['test_natural']}  "
          f"(class balance: {df_natural['TARGET'].mean():.4f}, population rate: {y.mean():.4f})")

## 5. Home Credit Default Risk

Kaggle setup: click **Add Data** in the right-hand panel, search `home-credit-default-risk`, add it.
It will mount at `/kaggle/input/home-credit-default-risk/`.


In [ ]:
print("Loading Home Credit from /kaggle/input/ ...")
hc_path = "/kaggle/input/home-credit-default-risk/application_train.csv"

if os.path.exists(hc_path):
    hc_df = pd.read_csv(hc_path)
    if "SK_ID_CURR" in hc_df.columns:
        n_before = hc_df.shape[1]
        hc_df = hc_df.drop(columns=["SK_ID_CURR"])
        print(f"Dropped SK_ID_CURR (customer ID, no predictive content; "
              f"{n_before} -> {hc_df.shape[1]} columns). v7 audit: this ID leaked into "
              f"the feature matrix in earlier runs; permutation-checked harmless to AUC "
              f"(delta < 0.0002) but dropped for hygiene since it is not a real feature.")
    process_and_save(hc_df, target_col="TARGET", dataset_name="home_credit")
else:
    print(f"ERROR: file not found at {hc_path}")
    print("Add the 'Home Credit Default Risk' dataset to this notebook via 'Add Data'.")

## 6. Taiwan Credit (Default of Credit Card Clients)

Tries `ucimlrepo` first; falls back to the raw UCI zip if that fails.


In [ ]:
print("Downloading Taiwan Credit dataset...")
from ucimlrepo import fetch_ucirepo

try:
    taiwan = fetch_ucirepo(id=350)
    X_tw = taiwan.data.features
    y_tw = taiwan.data.targets

    tw_df = pd.concat([X_tw, y_tw], axis=1)
    target_name = y_tw.columns[0]
    tw_df[target_name] = tw_df[target_name].astype(int)

    process_and_save(tw_df, target_col=target_name, dataset_name="taiwan")

except Exception as e:
    print(f"ucimlrepo failed: {e}")
    print("Using the official UCI zip fallback...")

    zip_url = "https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip"
    zip_path = "/tmp/taiwan_credit.zip"
    extract_dir = "/tmp/taiwan_credit"

    try:
        urllib.request.urlretrieve(zip_url, zip_path)
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_dir)

        tw_df = pd.read_excel(f"{extract_dir}/default of credit card clients.xls", header=1)
        if "ID" in tw_df.columns:
            tw_df = tw_df.drop(columns=["ID"])

        process_and_save(tw_df, target_col="default payment next month", dataset_name="taiwan")
    except Exception as e2:
        print(f"Fallback failed: {e2}")


## 7. Give Me Some Credit

Kaggle setup: **Add Data**, search `GiveMeSomeCredit`, add it.
It will mount at `/kaggle/input/GiveMeSomeCredit/`.


In [ ]:
print("Loading Give Me Some Credit from /kaggle/input/ ...")
gmsc_path = "/kaggle/input/GiveMeSomeCredit/cs-training.csv"

if os.path.exists(gmsc_path):
    gmsc_df = pd.read_csv(gmsc_path, index_col=0)
    gmsc_df.columns = gmsc_df.columns.str.replace(r"[\-\.\s]", "_", regex=True)
    process_and_save(gmsc_df, target_col="SeriousDlqin2yrs", dataset_name="gmsc")
else:
    print(f"ERROR: file not found at {gmsc_path}")
    print("Add the 'GiveMeSomeCredit' dataset to this notebook via 'Add Data'.")


## 8. Sanity check

Re-loads every saved file and reprints shapes / class balance, so a bad split is caught here
rather than three notebooks downstream.


In [ ]:
print(f"{'dataset':<14}{'split':<14}{'rows':>8}{'cols':>6}{'pos_rate':>10}")
for ds in ["home_credit", "taiwan", "gmsc"]:
    for split in ["train", "val", "test_balanced", "test_natural"]:
        p = f"{Config.OUTPUT_DIR}/{ds}_{split}.parquet"
        if not os.path.exists(p):
            print(f"{ds:<14}{split:<14}{'MISSING':>8}")
            continue
        d = pd.read_parquet(p)
        print(f"{ds:<14}{split:<14}{d.shape[0]:>8}{d.shape[1]:>6}{d['TARGET'].mean():>10.4f}")

print("\nDone. Upload /kaggle/working as a Kaggle Dataset named 'xai-credit-preprocessed'.")
